In [0]:
-- Let's create a Catalog for our project

CREATE CATALOG analytics_engineering;

-- Let's create a Schema for our project

CREATE SCHEMA analytics_engineering.bronze;

-- Let's create a Volume for our project

CREATE VOLUME analytics_engineering.bronze.raw_data;

In [0]:
-- Let's create Bronze Tables that will hold the raw data stored in the volume

-- Bronze Products Table

CREATE TABLE analytics_engineering.bronze.products_raw (
  product_id STRING,
  product_name STRING,
  category STRING,
  price STRING
);

COPY INTO analytics_engineering.bronze.products_raw
FROM '/Volumes/analytics_engineering/bronze/raw_data/Products.txt'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true');

-- Bronze customers Table

CREATE TABLE analytics_engineering.bronze.customers_raw (
  customer_id STRING,
  first_name STRING,
  last_name STRING,
  email STRING,
  signup_date STRING
);

COPY INTO analytics_engineering.bronze.customers_raw
FROM '/Volumes/analytics_engineering/bronze/raw_data/Customers.txt'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true');

-- Bronze Orders Table

CREATE TABLE analytics_engineering.bronze.orders_raw (
  order_id STRING,
  customer_id STRING,
  order_date STRING,
  status STRING
);

COPY INTO analytics_engineering.bronze.orders_raw
FROM '/Volumes/analytics_engineering/bronze/raw_data/Orders.txt'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true');

-- Bronze order_items Table

CREATE TABLE analytics_engineering.bronze.order_items_raw (
  order_item_id STRING,
  order_id STRING,
  product_id STRING,
  quantity STRING
);

COPY INTO analytics_engineering.bronze.order_items_raw
FROM '/Volumes/analytics_engineering/bronze/raw_data/Order_Items.txt'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true');



In [0]:
--Let's inspect all the Bronze tables

SELECT * FROM analytics_engineering.bronze.products_raw;

SELECT * FROM analytics_engineering.bronze.customers_raw;

SELECT * FROM analytics_engineering.bronze.orders_raw;

SELECT * FROM analytics_engineering.bronze.order_items_raw;





In [0]:
--Let's create the silver Schema

CREATE SCHEMA analytics_engineering.silver;

---Let's create the  DQ Results Table

CREATE TABLE analytics_engineering.silver.dq_results ( 
    rule_name STRING,
    table_name STRING,
    severity STRING,
    failed_count INT,
    sample_record STRING,
    run_timestamp TIMESTAMP
);

In [0]:
-- Universal SQL Pattern Statement for Logging Data Quality Rule Failures

--- For example: Logging Negative Prices (High Severity) into the DQ Results Table

INSERT INTO analytics_engineering.silver.dq_results
SELECT
    'negative_price' AS rule_name,
    'products_raw' AS table_name,
    'high' AS severity,
    COUNT(*) AS failed_count,
    TO_JSON(STRUCT(
        FIRST(product_id),
        FIRST(product_name),
        FIRST(category),
        FIRST(price)
    )) AS sample_record,
    current_timestamp() AS run_timestamp
FROM analytics_engineering.bronze.products_raw
WHERE CAST(price AS DOUBLE) < 0;

select * from analytics_engineering.silver.dq_results;

---Missing Order Date (High Severity) into the DQ Results Table

INSERT INTO analytics_engineering.silver.dq_results
SELECT
    'missing_order_date' AS rule_name,
    'orders_raw' AS table_name,
    'high' AS severity,
    COUNT(*) AS failed_count,
    TO_JSON(STRUCT(
        FIRST(order_id),
        FIRST(customer_id),
        FIRST(order_date),
        FIRST(status)
    )) AS sample_record,
    current_timestamp() AS run_timestamp
FROM analytics_engineering.bronze.orders_raw
WHERE order_date IS NULL;

select * from analytics_engineering.silver.dq_results;

In [0]:

---Let's create cleaned Silver layer for Customers table called customers_silver

CREATE OR REPLACE TABLE analytics_engineering.silver.customers_silver AS
WITH cleaned AS (
    SELECT
        CAST(customer_id AS INT) AS customer_id,
        first_name,
        last_name,
        CAST(signup_date AS DATE) AS signup_date,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY signup_date DESC
        ) AS rn
    FROM analytics_engineering.bronze.customers_raw
)
SELECT
    customer_id,
    first_name,
    last_name,
    signup_date
FROM cleaned
WHERE rn = 1                              -- enforce unique customer_id
  AND first_name IS NOT NULL              -- no missing first name
  AND last_name IS NOT NULL               -- no missing last name
  AND signup_date <= current_date();      -- no future signup dates

  select * from analytics_engineering.silver.customers_silver;


In [0]:
---Let's create cleaned Silver layer for Products table called Products_silver

CREATE OR REPLACE TABLE analytics_engineering.silver.products_silver AS
WITH cleaned AS (
    SELECT
        CAST(product_id AS INT) AS product_id,
        product_name,
        category,
        CAST(price AS DOUBLE) AS price,
        ROW_NUMBER() OVER (
            PARTITION BY product_id
            ORDER BY price DESC
        ) AS rn
    FROM analytics_engineering.bronze.products_raw
)
SELECT
    product_id,
    product_name,
    category,
    price
FROM cleaned
WHERE rn = 1                          -- enforce unique product_id
  AND product_id IS NOT NULL          -- product_id must not be NULL
  AND product_name IS NOT NULL        -- product_name must not be NULL
  AND TRIM(product_name) <> ''        -- product_name must not be empty
  AND price > 0;                      -- price must be greater than zero

  select * from analytics_engineering.silver.products_silver;

In [0]:
---Let's create cleaned Silver layer for Orders table called Orders_silver

CREATE OR REPLACE TABLE analytics_engineering.silver.orders_silver AS
WITH cleaned AS (
    SELECT
        CAST(order_id AS INT) AS order_id,
        CAST(customer_id AS INT) AS customer_id,
        CAST(order_date AS DATE) AS order_date,
        status,
        ROW_NUMBER() OVER (
            PARTITION BY order_id
            ORDER BY order_date DESC
        ) AS rn
    FROM analytics_engineering.bronze.orders_raw
),
validated AS (
    SELECT
        c.order_id,
        c.customer_id,
        c.order_date,
        c.status
    FROM cleaned c
    INNER JOIN analytics_engineering.silver.customers_silver s
        ON c.customer_id = s.customer_id   -- enforce FK
    WHERE c.rn = 1                          -- unique order_id
      AND c.order_id IS NOT NULL            -- must not be NULL
      AND c.order_date IS NOT NULL          -- must not be NULL
      AND c.order_date <= current_date()    -- must not be in the future
)
SELECT * FROM validated;

   select * from analytics_engineering.silver.orders_silver;

In [0]:
---Let's create cleaned Silver layer for Orders_Items table called Order_Items_silver

CREATE OR REPLACE TABLE analytics_engineering.silver.order_items_silver AS
WITH cleaned AS (
    SELECT
        CAST(order_item_id AS INT) AS order_item_id,
        CAST(order_id AS INT) AS order_id,
        CAST(product_id AS INT) AS product_id,
        CAST(quantity AS INT) AS quantity,
        ROW_NUMBER() OVER (
            PARTITION BY order_item_id
            ORDER BY quantity DESC
        ) AS rn
    FROM analytics_engineering.bronze.order_items_raw
),
validated AS (
    SELECT
        c.order_item_id,
        c.order_id,
        c.product_id,
        c.quantity
    FROM cleaned c
    INNER JOIN analytics_engineering.silver.orders_silver o
        ON c.order_id = o.order_id          -- enforce FK to orders
    INNER JOIN analytics_engineering.silver.products_silver p
        ON c.product_id = p.product_id      -- enforce FK to products
    WHERE c.rn = 1                           -- unique order_item_id
      AND c.order_item_id IS NOT NULL        -- must not be NULL
      AND c.quantity > 0                     -- quantity must be > 0
)
SELECT * FROM validated;

select * from analytics_engineering.silver.order_items_silver;

In [0]:
---Let's create quarantine_products table for products_raw

CREATE TABLE analytics_engineering.silver.quarantine_products (
    original_record STRING,
    rule_name STRING,
    failure_reason STRING,
    severity STRING,
    table_name STRING,
    run_timestamp TIMESTAMP
);

--- Insert Invalid rows

WITH numbered AS (
    SELECT
        product_id,
        product_name,
        category,
        price,
        ROW_NUMBER() OVER (
            PARTITION BY product_id
            ORDER BY CAST(price AS DOUBLE) DESC
        ) AS rn
    FROM analytics_engineering.bronze.products_raw
)

INSERT INTO analytics_engineering.silver.quarantine_products
SELECT
    TO_JSON(STRUCT(
        product_id,
        product_name,
        category,
        price
    )) AS original_record,
    rule_name,
    failure_reason,
    'high' AS severity,
    'products_raw' AS table_name,
    current_timestamp() AS run_timestamp
FROM (

    -- Missing product_id
    SELECT
        product_id,
        product_name,
        category,
        price,
        'missing_product_id' AS rule_name,
        'product_id must not be NULL' AS failure_reason
    FROM analytics_engineering.bronze.products_raw
    WHERE product_id IS NULL

    UNION ALL

    -- Missing product_name
    SELECT
        product_id,
        product_name,
        category,
        price,
        'missing_product_name',
        'product_name must not be NULL'
    FROM analytics_engineering.bronze.products_raw
    WHERE product_name IS NULL OR TRIM(product_name) = ''

    UNION ALL

    -- Price <= 0
    SELECT
        product_id,
        product_name,
        category,
        price,
        'invalid_price',
        'price must be greater than 0'
    FROM analytics_engineering.bronze.products_raw
    WHERE CAST(price AS DOUBLE) <= 0

    UNION ALL

    -- Duplicate product_id (keep only the most recent / highest price)
    SELECT
        product_id,
        product_name,
        category,
        price,
        'duplicate_product_id',
        'product_id must be unique'
    FROM numbered
    WHERE rn > 1

);

select * from analytics_engineering.silver.quarantine_products;

In [0]:

--- Lets create quarantine_customers table for customers_raw

CREATE TABLE IF NOT EXISTS analytics_engineering.silver.quarantine_customers (
    original_record STRING,
    rule_name STRING,
    failure_reason STRING,
    severity STRING,
    table_name STRING,
    run_timestamp TIMESTAMP
);

--- Insert Invalid rows

WITH numbered AS (
    SELECT
        customer_id,
        first_name,
        last_name,
        signup_date,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY CAST(signup_date AS DATE) DESC
        ) AS rn
    FROM analytics_engineering.bronze.customers_raw
)

INSERT INTO analytics_engineering.silver.quarantine_customers
SELECT
    TO_JSON(STRUCT(
        customer_id,
        first_name,
        last_name,
        signup_date
    )) AS original_record,
    rule_name,
    failure_reason,
    'high' AS severity,
    'customers_raw' AS table_name,
    current_timestamp() AS run_timestamp
FROM (

    -- Missing customer_id
    SELECT
        customer_id,
        first_name,
        last_name,
        signup_date,
        'missing_customer_id' AS rule_name,
        'customer_id must not be NULL' AS failure_reason
    FROM analytics_engineering.bronze.customers_raw
    WHERE customer_id IS NULL

    UNION ALL

    -- Missing first_name
    SELECT
        customer_id,
        first_name,
        last_name,
        signup_date,
        'missing_first_name',
        'first_name must not be NULL'
    FROM analytics_engineering.bronze.customers_raw
    WHERE first_name IS NULL

    UNION ALL

    -- Missing last_name
    SELECT
        customer_id,
        first_name,
        last_name,
        signup_date,
        'missing_last_name',
        'last_name must not be NULL'
    FROM analytics_engineering.bronze.customers_raw
    WHERE last_name IS NULL

    UNION ALL

    -- Future signup_date
    SELECT
        customer_id,
        first_name,
        last_name,
        signup_date,
        'future_signup_date',
        'signup_date cannot be in the future'
    FROM analytics_engineering.bronze.customers_raw
    WHERE CAST(signup_date AS DATE) > current_date()

    UNION ALL

    -- Duplicate customer_id (keep only the most recent)
    SELECT
        customer_id,
        first_name,
        last_name,
        signup_date,
        'duplicate_customer_id',
        'customer_id must be unique'
    FROM numbered
    WHERE rn > 1

);

select * from analytics_engineering.silver.quarantine_customers;


In [0]:
---Let's create quarantine_orders table for orders_raw

CREATE TABLE IF NOT EXISTS analytics_engineering.silver.quarantine_orders (
    original_record STRING,
    rule_name STRING,
    failure_reason STRING,
    severity STRING,
    table_name STRING,
    run_timestamp TIMESTAMP
);

--- Insert Invalid rows

WITH numbered AS (
    SELECT
        order_id,
        customer_id,
        order_date,
        status,
        ROW_NUMBER() OVER (
            PARTITION BY order_id
            ORDER BY order_date DESC
        ) AS rn
    FROM analytics_engineering.bronze.orders_raw
)

INSERT INTO analytics_engineering.silver.quarantine_orders
SELECT
    TO_JSON(STRUCT(
        order_id,
        customer_id,
        order_date,
        status
    )) AS original_record,
    rule_name,
    failure_reason,
    'high' AS severity,
    'orders_raw' AS table_name,
    current_timestamp() AS run_timestamp
FROM (

    -- Missing order_id
    SELECT
        order_id,
        customer_id,
        order_date,
        status,
        'missing_order_id' AS rule_name,
        'order_id must not be NULL' AS failure_reason
    FROM analytics_engineering.bronze.orders_raw
    WHERE order_id IS NULL

    UNION ALL

    -- Missing order_date
    SELECT
        order_id,
        customer_id,
        order_date,
        status,
        'missing_order_date',
        'order_date must not be NULL'
    FROM analytics_engineering.bronze.orders_raw
    WHERE order_date IS NULL

    UNION ALL

    -- Future order_date
    SELECT
        order_id,
        customer_id,
        order_date,
        status,
        'future_order_date',
        'order_date cannot be in the future'
    FROM analytics_engineering.bronze.orders_raw
    WHERE CAST(order_date AS DATE) > current_date()

    UNION ALL

    -- Invalid customer_id (FK violation)
    SELECT
        o.order_id,
        o.customer_id,
        o.order_date,
        o.status,
        'invalid_customer_id',
        'customer_id does not exist in customers_silver'
    FROM analytics_engineering.bronze.orders_raw o
    LEFT JOIN analytics_engineering.silver.customers_silver s
        ON CAST(o.customer_id AS INT) = s.customer_id
    WHERE s.customer_id IS NULL

    UNION ALL

    -- Duplicate order_id (all but the most recent)
    SELECT
        order_id,
        customer_id,
        order_date,
        status,
        'duplicate_order_id',
        'order_id must be unique'
    FROM numbered
    WHERE rn > 1

);


select * from analytics_engineering.silver.quarantine_orders;


In [0]:
---Let's create quarantine_order_items table for order_items_raw

CREATE TABLE IF NOT EXISTS analytics_engineering.silver.quarantine_order_items (
    original_record STRING,
    rule_name STRING,
    failure_reason STRING,
    severity STRING,
    table_name STRING,
    run_timestamp TIMESTAMP
);

--- Insert Invalid rows

WITH numbered AS (
    SELECT
        order_item_id,
        order_id,
        product_id,
        quantity,
        ROW_NUMBER() OVER (
            PARTITION BY order_item_id
            ORDER BY quantity DESC
        ) AS rn
    FROM analytics_engineering.bronze.order_items_raw
)

INSERT INTO analytics_engineering.silver.quarantine_order_items
SELECT
    TO_JSON(STRUCT(
        order_item_id,
        order_id,
        product_id,
        quantity
    )) AS original_record,
    rule_name,
    failure_reason,
    'high' AS severity,
    'order_items_raw' AS table_name,
    current_timestamp() AS run_timestamp
FROM (

    -- Missing order_item_id
    SELECT
        order_item_id,
        order_id,
        product_id,
        quantity,
        'missing_order_item_id' AS rule_name,
        'order_item_id must not be NULL' AS failure_reason
    FROM analytics_engineering.bronze.order_items_raw
    WHERE order_item_id IS NULL

    UNION ALL

    -- Quantity <= 0
    SELECT
        order_item_id,
        order_id,
        product_id,
        quantity,
        'invalid_quantity',
        'quantity must be greater than 0'
    FROM analytics_engineering.bronze.order_items_raw
    WHERE CAST(quantity AS INT) <= 0

    UNION ALL

    -- Invalid order_id (FK violation)
    SELECT
        oi.order_item_id,
        oi.order_id,
        oi.product_id,
        oi.quantity,
        'invalid_order_id',
        'order_id does not exist in orders_silver'
    FROM analytics_engineering.bronze.order_items_raw oi
    LEFT JOIN analytics_engineering.silver.orders_silver o
        ON CAST(oi.order_id AS INT) = o.order_id
    WHERE o.order_id IS NULL

    UNION ALL

    -- Invalid product_id (FK violation)
    SELECT
        oi.order_item_id,
        oi.order_id,
        oi.product_id,
        oi.quantity,
        'invalid_product_id',
        'product_id does not exist in products_silver'
    FROM analytics_engineering.bronze.order_items_raw oi
    LEFT JOIN analytics_engineering.silver.products_silver p
        ON CAST(oi.product_id AS INT) = p.product_id
    WHERE p.product_id IS NULL

    UNION ALL

    -- Duplicate order_item_id (keep only the highest quantity)
    SELECT
        order_item_id,
        order_id,
        product_id,
        quantity,
        'duplicate_order_item_id',
        'order_item_id must be unique'
    FROM numbered
    WHERE rn > 1

);

select * from analytics_engineering.silver.quarantine_order_items;


In [0]:
---Let's create the Gold Schema

CREATE SCHEMA analytics_engineering.gold;

In [0]:
--- Let's create the dim_customers table

CREATE OR REPLACE TABLE analytics_engineering.gold.dim_customers AS
SELECT
    monotonically_increasing_id() AS customer_sk,   -- surrogate key
    customer_id,                                     -- natural key
    first_name,
    last_name,
    signup_date
FROM analytics_engineering.silver.customers_silver;

Select * from analytics_engineering.gold.dim_customers

In [0]:
--- Let's create the dim_products table

CREATE OR REPLACE TABLE analytics_engineering.gold.dim_products AS
SELECT
    monotonically_increasing_id() AS product_sk,
    product_id,
    product_name,
    category,
    price
FROM analytics_engineering.silver.products_silver;

Select * from analytics_engineering.gold.dim_products;

In [0]:
--- Let's create fact_orders table

CREATE OR REPLACE TABLE analytics_engineering.gold.fact_orders AS
SELECT
    o.order_id,
    c.customer_sk,
    o.order_date,
    o.status
FROM analytics_engineering.silver.orders_silver o
LEFT JOIN analytics_engineering.gold.dim_customers c
    ON o.customer_id = c.customer_id;

Select * from analytics_engineering.gold.fact_orders;

In [0]:
--- Let's create fact_order_items table

CREATE OR REPLACE TABLE analytics_engineering.gold.fact_order_items AS
SELECT
    oi.order_item_id,
    fo.order_id,
    dp.product_sk,
    oi.quantity,
    dp.price,
    (oi.quantity * dp.price) AS total_amount
FROM analytics_engineering.silver.order_items_silver oi
LEFT JOIN analytics_engineering.gold.fact_orders fo
    ON oi.order_id = fo.order_id
LEFT JOIN analytics_engineering.gold.dim_products dp
    ON oi.product_id = dp.product_id;

SELECT * FROM analytics_engineering.gold.fact_order_items